In [ ]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup, NavigableString

header = {"User-Agent": "Mozilla/5.0"}

good_spans = {
    "Stage Name",
    "Stage / Birth Name"
    "Birth Name",
    "Former Member",
    "Former Members",
    "Hometown",
    "Was born",
    "raised",
    "Nationality"
}

bad_keywords = (
    "facts", "note", "emoji", "position",
    "birthday", "height", "weight", "blood type"
)

# код работает только потому что в груплист были добавлены ССЫЛКИ 

csv_input = r"grouplist.csv"
csv_output = r"parsed_kprofiles.csv"

df_groups = pd.read_csv(csv_input)

required_cols = {"group_name", "link"}
if not required_cols.issubset(df_groups.columns):
    raise ValueError(f"в CSV должны быть столбцы: {required_cols}")


df_groups = df_groups.dropna(subset=["link"])

session = requests.Session()
session.headers.update(header)

rows = []

for _, row in df_groups.iterrows():
    group_name = str(row["group_name"]).strip()
    url = str(row["link"]).strip()

    print(f"\nВ процессе: {group_name}")
    print(f"URL: {url}")

    try:
        response = session.get(url, timeout=15)
        if response.status_code != 200:
            print(f"Ошибка {response.status_code}: {group_name}")
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        content = soup.select_one("div.entry-content")
        if not content:
            print("Не найден div.entry-content")
            continue

        current_member = None

        for p in content.find_all("p"):
            strong = p.find("strong")

            if strong:
                name = strong.get_text(strip=True)

                if len(name) < 50:
                    if any(k in name.lower() for k in bad_keywords):
                        pass
                    elif "member profiles" in name.lower():
                        pass
                    else:
                        current_member = name

            for span in p.find_all("span"):
                label = span.get_text(strip=True).replace(":", "")

                if label not in good_spans:
                    continue

                sib = span.next_sibling
                value = ""

                if isinstance(sib, NavigableString):
                    value = sib.strip()

                if not value or not current_member:
                    continue

                rows.append({
                    "group": group_name,
                    "name": current_member,
                    "span_label": label,
                    "span_value": value
                })

        time.sleep(1)

    except Exception as e:
        print(f"Ошибка при обработке {group_name}: {e}")
        continue

df_result = pd.DataFrame(rows)

print("\nПример результата:")
if not df_result.empty:
    print(df_result.sample(min(15, len(df_result))))
else:
    print("Датафрейм пуст")

print("\nВсего строк:", len(df_result))

df_result.to_csv(csv_output, index=False, encoding="utf-8")
print(f"\nСохранено в {csv_output}")
